# Generate Folds (Python port)

Python reimplementation of the OpenML `EvaluationEngine` split generators
(Java sources under `foldgenerators/` and `generatefolds/GenerateFolds.java`).

Each generator produces a **splits table**: one row per `(instance, repeat, fold, [sample])`
assignment, telling OpenML whether that original instance belongs to `TRAIN` or `TEST`
for that combination.

The splits table schema mirrors the Java `ArffMapping`:

| column   | meaning                                                          |
|----------|------------------------------------------------------------------|
| `type`   | `'TRAIN'` or `'TEST'`                                            |
| `rowid`  | original 0-based row index in the dataset                        |
| `repeat` | repeat index (0-based)                                           |
| `fold`   | fold index (0-based)                                             |
| `sample` | subsample index (learning-curve tasks only)                      |

Splitting primitives come from `scikit-learn`; everything else is `pandas`.
Dataset download reuses the existing `src.helpers` utilities.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
from typing import Optional

import arff
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    KFold,
    LeaveOneOut,
    ShuffleSplit,
    StratifiedKFold,
)

from src.helpers import get_data_and_meta_information_from_did
from models import DatasetDownloadInfo

## 1. Load a dataset into a pandas DataFrame

`get_data_and_meta_information_from_did` (already in `src/helpers.py`) downloads the
ARFF for a given dataset id and returns its temp file path plus the declared target
attribute. We wrap `liac-arff` to turn that into a DataFrame, preserving the **file
order** so that row indices are stable `rowid`s.

In [2]:
def load_dataset(did: int) -> tuple[pd.DataFrame, Optional[str]]:
    """Download an OpenML dataset by id and return (DataFrame, target_attribute)."""
    info: DatasetDownloadInfo = get_data_and_meta_information_from_did(did)

    with open(info.file_path, "r", encoding="utf-8", errors="replace") as f:
        payload = arff.load(f)

    attributes = payload["attributes"]
    columns = [name for name, _ in attributes]

    df = pd.DataFrame(payload["data"], columns=columns)

    # Encode nominal columns as pandas categoricals so dtype inspection
    # (used for stratification decisions) behaves intuitively.
    for name, type_spec in attributes:
        if isinstance(type_spec, list):
            df[name] = pd.Categorical(df[name], categories=type_spec)

    return df, info.default_target_attribute

## 2. Estimation-procedure configuration

The Java side dispatches on `EstimationProcedureType` and reads `folds`, `repeats`,
`percentage` from the procedure. We mirror that with a small dataclass and enum so
the dispatcher below looks one-to-one with `GenerateFolds.java`.

In [3]:
class EstimationProcedureType(str, Enum):
    CROSSVALIDATION = "CROSSVALIDATION"
    HOLDOUT = "HOLDOUT"
    HOLDOUT_ORDERED = "HOLDOUT_ORDERED"
    LEAVEONEOUT = "LEAVEONEOUT"
    TESTONTRAININGDATA = "TESTONTRAININGDATA"
    LEARNINGCURVE_CV = "LEARNINGCURVE_CV"


@dataclass(frozen=True)
class EstimationProcedure:
    type: EstimationProcedureType
    folds: Optional[int] = None
    repeats: Optional[int] = None
    percentage: Optional[float] = None  # test-set size, 0..100


def _is_nominal(df: pd.DataFrame, target: Optional[str]) -> bool:
    """Match Weka's `classAttribute().isNominal()` check for stratification."""
    if target is None or target not in df.columns:
        return False
    dtype = df[target].dtype
    return isinstance(dtype, pd.CategoricalDtype) or dtype == object


def _empty_splits(with_sample: bool = False) -> pd.DataFrame:
    cols = ["type", "rowid", "repeat", "fold"]
    if with_sample:
        cols.append("sample")
    return pd.DataFrame({c: pd.Series(dtype="int64" if c != "type" else "object") for c in cols})

## 3. Holdout splits

Source: `HoldoutSplitsGenerator.java`. For each repeat, shuffle the data with the
seed, then take the first `round(N * percentage / 100)` instances as `TEST`, the
rest as `TRAIN`. `sklearn.model_selection.ShuffleSplit` does exactly this and gives
us `repeats` independent shuffles in one call.

In [4]:
def holdout_splits(
    df: pd.DataFrame,
    procedure: EstimationProcedure,
    seed: int = 1,
) -> pd.DataFrame:
    n = len(df)
    test_size = procedure.percentage / 100.0
    ss = ShuffleSplit(
        n_splits=procedure.repeats,
        test_size=test_size,
        random_state=seed,
    )

    rows: list[tuple] = []
    # rowid is the ORIGINAL row index, so we split an index array, not df rows.
    index = np.arange(n)
    for repeat, (train_idx, test_idx) in enumerate(ss.split(index)):
        for i in train_idx:
            rows.append(("TRAIN", int(i), repeat, 0))
        for i in test_idx:
            rows.append(("TEST", int(i), repeat, 0))

    return pd.DataFrame(rows, columns=["type", "rowid", "repeat", "fold"])

## 4. Holdout-ordered splits

Source: `HoldoutOrderedSplitsGenerator.java`. No shuffling — the first
`(100 - percentage)%` of instances (in file order) become `TRAIN`, the tail becomes
`TEST`. Exactly one repeat and one fold.

In [5]:
def holdout_ordered_splits(
    df: pd.DataFrame,
    procedure: EstimationProcedure,
) -> pd.DataFrame:
    n = len(df)
    test_size = n * procedure.percentage / 100.0
    threshold = n - test_size  # rows at index <= threshold are TRAIN

    rows = [
        ("TRAIN" if i <= threshold else "TEST", i, 0, 0)
        for i in range(n)
    ]
    return pd.DataFrame(rows, columns=["type", "rowid", "repeat", "fold"])

## 5. Cross-validation splits

Source: `CrossValidationSplitsGenerator.java`. For each repeat:
1. shuffle the dataset with the seed;
2. **stratify** if the target is nominal;
3. carve into `folds` slices and emit every instance as `TRAIN` or `TEST`
   depending on whether it belongs to fold *f*.

`StratifiedKFold(shuffle=True)` is the sklearn equivalent of Weka's
`randomize` + `stratify` + `trainCV/testCV`. We instantiate one splitter per repeat
with a distinct child seed so each repeat gets a fresh permutation (mirroring the
Java loop which draws from the same `Random` consecutively).

In [6]:
def crossvalidation_splits(
    df: pd.DataFrame,
    procedure: EstimationProcedure,
    target: Optional[str] = None,
    seed: int = 1,
) -> pd.DataFrame:
    n = len(df)
    stratify = _is_nominal(df, target)
    y = df[target].to_numpy() if stratify else None
    index = np.arange(n)

    rows: list[tuple] = []
    for repeat in range(procedure.repeats):
        if stratify:
            splitter = StratifiedKFold(
                n_splits=procedure.folds,
                shuffle=True,
                random_state=seed + repeat,
            )
            splits = splitter.split(index, y)
        else:
            splitter = KFold(
                n_splits=procedure.folds,
                shuffle=True,
                random_state=seed + repeat,
            )
            splits = splitter.split(index)

        for fold, (train_idx, test_idx) in enumerate(splits):
            for i in train_idx:
                rows.append(("TRAIN", int(i), repeat, fold))
            for i in test_idx:
                rows.append(("TEST", int(i), repeat, fold))

    return pd.DataFrame(rows, columns=["type", "rowid", "repeat", "fold"])

## 6. Leave-one-out splits

Source: `LeaveOneOutSplitsGenerator.java`. For each fold *f* in `0..N-1`, instance
*f* is `TEST` and all others are `TRAIN`. The output has `N * N` rows. sklearn's
`LeaveOneOut` yields the train/test index pairs directly.

In [7]:
def leave_one_out_splits(df: pd.DataFrame) -> pd.DataFrame:
    index = np.arange(len(df))
    rows: list[tuple] = []
    for fold, (train_idx, test_idx) in enumerate(LeaveOneOut().split(index)):
        for i in train_idx:
            rows.append(("TRAIN", int(i), 0, fold))
        for i in test_idx:
            rows.append(("TEST", int(i), 0, fold))
    return pd.DataFrame(rows, columns=["type", "rowid", "repeat", "fold"])

## 7. Test-on-training-data splits

Source: `TrainOnTestSplitsGenerator.java`. Every instance appears twice in fold 0 —
once as `TRAIN` and once as `TEST`. Trains and tests on the same data; used for
in-sample evaluation. Output size is `2 * N`.

In [8]:
def train_on_test_splits(df: pd.DataFrame) -> pd.DataFrame:
    rows: list[tuple] = []
    for i in range(len(df)):
        rows.append(("TEST", i, 0, 0))
        rows.append(("TRAIN", i, 0, 0))
    return pd.DataFrame(rows, columns=["type", "rowid", "repeat", "fold"])

## 8. Learning-curve splits

Source: `CrossValidationSplitsGenerator.generate_learningcurve` plus the sample-size
helpers in `FoldGeneratorBase.java`. Same CV skeleton as §5, but inside each fold
the **training** side is subsampled at geometrically growing sizes
`2^(6 + 0.5 * s)` (capped at the full train size), and each subsample is emitted
under its own `sample` index. The full test fold is repeated for every sample.

In [9]:
def sample_size(number: int, train_size: int) -> int:
    """2^(6 + 0.5 * number), capped at train_size (FoldGeneratorBase.sampleSize)."""
    return int(min(train_size, round(2 ** (6 + number * 0.5))))


def num_samples(train_size: int) -> int:
    """Number of subsamples for a training set of this size (incl. the full set)."""
    i = 0
    while sample_size(i, train_size) < train_size:
        i += 1
    return i + 1

In [10]:
def learning_curve_splits(
    df: pd.DataFrame,
    procedure: EstimationProcedure,
    target: Optional[str] = None,
    seed: int = 1,
) -> pd.DataFrame:
    n = len(df)
    stratify = _is_nominal(df, target)
    y = df[target].to_numpy() if stratify else None
    index = np.arange(n)

    rows: list[tuple] = []
    for repeat in range(procedure.repeats):
        if stratify:
            splitter = StratifiedKFold(
                n_splits=procedure.folds,
                shuffle=True,
                random_state=seed + repeat,
            )
            splits = splitter.split(index, y)
        else:
            splitter = KFold(
                n_splits=procedure.folds,
                shuffle=True,
                random_state=seed + repeat,
            )
            splits = splitter.split(index)

        for fold, (train_idx, test_idx) in enumerate(splits):
            train_size = len(train_idx)
            for s in range(num_samples(train_size)):
                k = sample_size(s, train_size)
                # first k (already-shuffled) training rows at this sample size
                for i in train_idx[:k]:
                    rows.append(("TRAIN", int(i), repeat, fold, s))
                for i in test_idx:
                    rows.append(("TEST", int(i), repeat, fold, s))

    return pd.DataFrame(
        rows,
        columns=["type", "rowid", "repeat", "fold", "sample"],
    )

## 9. Dispatcher

One-to-one with `GenerateFolds.java`: download the dataset, then route to the
matching generator based on `EstimationProcedureType`. Returns the splits
DataFrame plus the loaded dataset (handy for inspection).

In [11]:
def generate_folds(
    did: int,
    procedure: EstimationProcedure,
    seed: int = 1,
) -> tuple[pd.DataFrame, pd.DataFrame, Optional[str]]:
    """Download dataset `did` and compute its splits table.

    Returns (splits, dataset, target_attribute).
    """
    df, target = load_dataset(did)

    if procedure.type is EstimationProcedureType.CROSSVALIDATION:
        splits = crossvalidation_splits(df, procedure, target=target, seed=seed)
    elif procedure.type is EstimationProcedureType.LEARNINGCURVE_CV:
        splits = learning_curve_splits(df, procedure, target=target, seed=seed)
    elif procedure.type is EstimationProcedureType.HOLDOUT:
        splits = holdout_splits(df, procedure, seed=seed)
    elif procedure.type is EstimationProcedureType.HOLDOUT_ORDERED:
        splits = holdout_ordered_splits(df, procedure)
    elif procedure.type is EstimationProcedureType.LEAVEONEOUT:
        splits = leave_one_out_splits(df)
    elif procedure.type is EstimationProcedureType.TESTONTRAININGDATA:
        splits = train_on_test_splits(df)
    else:
        raise ValueError(f"Unsupported procedure type: {procedure.type}")

    return splits, df, target

## 10. Serialize splits to OpenML ARFF

Java's `ArffMapping` emits a 4- or 5-column ARFF (`type`, `rowid`, `repeat`, `fold`,
optional `sample`). We reproduce that schema with `liac-arff` (already a project
dependency) so the output is byte-compatible with what the OpenML server accepts.

In [12]:
def splits_to_arff(splits: pd.DataFrame, relation: str = "splits") -> str:
    """Serialize a splits DataFrame to an OpenML-format ARFF string.

    Schema matches Java's `ArffMapping`: `type` is nominal {TRAIN, TEST},
    every other column is NUMERIC. Columns are emitted in declaration order
    regardless of DataFrame column order.
    """
    attributes: list[tuple[str, object]] = [("type", ["TRAIN", "TEST"])]
    for col in ("rowid", "repeat", "fold", "sample"):
        if col in splits.columns:
            attributes.append((col, "NUMERIC"))

    ordered = splits[[name for name, _ in attributes]]
    data = ordered.astype(object).to_numpy().tolist()

    return arff.dumps(
        {
            "relation": relation,
            "attributes": attributes,
            "data": data,
        }
    )


def save_splits_arff(
    splits: pd.DataFrame,
    path: str,
    relation: str = "splits",
) -> None:
    """Write a splits DataFrame to `path` as OpenML-format ARFF."""
    text = splits_to_arff(splits, relation=relation)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def arff_head(text: str, n: int = 15) -> str:
    """First n lines of an ARFF string — handy for notebook display."""
    return "\n".join(text.splitlines()[:n])

## 11. Demo — 10-fold CV on `did=61` (iris)

Stratified 10-fold CV, 1 repeat, seed 1. Expect `10 * N` rows where every rowid
appears exactly 9 times as `TRAIN` and once as `TEST`. We also serialize the
result to ARFF and show the first lines.

In [13]:
splits, df, target = generate_folds(
    did=61,
    procedure=EstimationProcedure(
        type=EstimationProcedureType.CROSSVALIDATION,
        folds=10,
        repeats=1,
    ),
    seed=1,
)

print(f"dataset: {len(df)} rows, target = {target!r}")
print(f"splits : {len(splits)} rows")

cv_arff = splits_to_arff(splits, relation="iris_splits")
save_splits_arff(splits, "iris_cv_splits.arff", relation="iris_splits")
print("\n--- ARFF head ---")
print(arff_head(cv_arff, n=12))

dataset: 150 rows, target = 'class'
splits : 1500 rows

--- ARFF head ---
@RELATION iris_splits

@ATTRIBUTE type {TRAIN, TEST}
@ATTRIBUTE rowid NUMERIC
@ATTRIBUTE repeat NUMERIC
@ATTRIBUTE fold NUMERIC

@DATA
TRAIN,0,0,0
TRAIN,1,0,0
TRAIN,2,0,0
TRAIN,3,0,0


In [14]:
# sanity: each rowid appears folds-1 times as TRAIN and once as TEST
pivot = splits.pivot_table(
    index="rowid", columns="type", values="fold", aggfunc="count", fill_value=0,
)
pivot["TRAIN"].value_counts(), pivot["TEST"].value_counts()

(TRAIN
 9    150
 Name: count, dtype: int64,
 TEST
 1    150
 Name: count, dtype: int64)

## 12. Demo — 66/33 holdout, 3 repeats

Holdout emits the same 4-column schema as CV; learning curve (§13) is the only
generator that adds a `sample` column.

In [15]:
splits_h, _, _ = generate_folds(
    did=61,
    procedure=EstimationProcedure(
        type=EstimationProcedureType.HOLDOUT,
        percentage=33,
        repeats=3,
    ),
    seed=1,
)
display(splits_h.groupby(["repeat", "type"]).size().unstack(fill_value=0))

holdout_arff = splits_to_arff(splits_h, relation="iris_holdout_splits")
print("--- ARFF head ---")
print(arff_head(holdout_arff, n=10))

type,TEST,TRAIN
repeat,,
0,50,100
1,50,100
2,50,100


--- ARFF head ---
@RELATION iris_holdout_splits

@ATTRIBUTE type {TRAIN, TEST}
@ATTRIBUTE rowid NUMERIC
@ATTRIBUTE repeat NUMERIC
@ATTRIBUTE fold NUMERIC

@DATA
TRAIN,39,0,0
TRAIN,36,0,0


## 13. Demo — learning curve (5-fold CV, subsampled training sets)

Same skeleton as §11 but the `TRAIN` rows per fold are partitioned by `sample`,
with sizes growing as `2^(6 + 0.5*s)`. This is the only generator that emits a
`sample` column, so the ARFF carries five attributes.

In [16]:
splits_lc, _, _ = generate_folds(
    did=61,
    procedure=EstimationProcedure(
        type=EstimationProcedureType.LEARNINGCURVE_CV,
        folds=5,
        repeats=1,
    ),
    seed=1,
)
# training-set size per (fold, sample) — should double roughly every two samples
display(
    splits_lc[splits_lc.type == "TRAIN"]
    .groupby(["fold", "sample"])
    .size()
    .unstack(fill_value=0)
)

lc_arff = splits_to_arff(splits_lc, relation="iris_learningcurve_splits")
save_splits_arff(splits_lc, "iris_lc_splits.arff", relation="iris_learningcurve_splits")
print("--- ARFF head (note the @ATTRIBUTE sample) ---")
print(arff_head(lc_arff, n=12))

sample,0,1,2
fold,,,
0,64,91,120
1,64,91,120
2,64,91,120
3,64,91,120
4,64,91,120


--- ARFF head (note the @ATTRIBUTE sample) ---
@RELATION iris_learningcurve_splits

@ATTRIBUTE type {TRAIN, TEST}
@ATTRIBUTE rowid NUMERIC
@ATTRIBUTE repeat NUMERIC
@ATTRIBUTE fold NUMERIC
@ATTRIBUTE sample NUMERIC

@DATA
TRAIN,0,0,0,0
TRAIN,1,0,0,0
TRAIN,2,0,0,0


## Caveat: RNG parity

Weka's `randomize` / `stratify` and sklearn's `ShuffleSplit` / `StratifiedKFold`
use different RNG implementations, so the **specific `rowid` assignments will not
byte-match** the ARFF files produced by the Java engine for the same `seed`.

The **procedure** (which instances form each train/test pair, the stratification,
the sample-size schedule for learning curves) is faithfully reproduced, so the
splits are statistically equivalent — just not bit-identical to OpenML's server
output. Matching Weka's permutation exactly would require porting Weka's
`Random` + stratification order, which is orthogonal to using sklearn.